In [1]:
import os
import json
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
from transformers import BlipProcessor, BlipForConditionalGeneration

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"PyTorch : {torch.__version__}")


c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0605 21:58:58.133000 20100 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch : 2.11.0+cu126


In [2]:
ROOT_DIR    = "../data_ready_for_kaggle"
TEST_PATH   = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR  = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR   = os.path.abspath("../saved_models_v3")
SAVE_DIR    = "./model_quantized/pytorch_int8"
RESULTS_DIR = "./results"

MAX_TEST_IMAGES = 200
MAX_LENGTH      = 64
BATCH_SIZE      = 16
NUM_WORKERS     = 0
WARMUP_STEPS    = 3

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cpu')
print(f"Device    : {device}")
print(f"Model dir : {MODEL_DIR}")


Device    : cpu
Model dir : c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\saved_models_v3


In [3]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        df = pd.read_csv(data_file)
        self.img_dir    = img_dir
        self.processor  = processor
        self.max_length = max_length
        self.image_groups = (
            df.groupby('image_file')['caption'].apply(list).reset_index()
        )

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, idx):
        row        = self.image_groups.iloc[idx]
        image_file = row['image_file']
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')
        enc = self.processor(
            images=image, return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc['image_file']   = image_file
        enc['raw_captions'] = row['caption']
        return enc


def collate_fn(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ['pixel_values']}
    out['image_file']   = [b['image_file']   for b in batch]
    out['raw_captions'] = [b['raw_captions'] for b in batch]
    return out


def build_references(dataloader):
    refs = {}
    for batch in dataloader:
        for img, caps in zip(batch['image_file'], batch['raw_captions']):
            refs[img] = caps
    return refs


def calculate_metrics(preds_dict, refs_dict):
    common = sorted(set(preds_dict) & set(refs_dict))
    preds  = [preds_dict[k] for k in common]
    refs   = [refs_dict[k]  for k in common]
    bleu   = evaluate.load('bleu')
    rouge  = evaluate.load('rouge')
    meteor = evaluate.load('meteor')
    return {
        'bleu4':  bleu.compute(predictions=preds, references=refs)['bleu'] * 100,
        'rougeL': rouge.compute(predictions=preds, references=refs)['rougeL'] * 100,
        'meteor': meteor.compute(predictions=preds, references=refs)['meteor'] * 100,
        'num_images': len(common),
    }


def run_generation(m, processor, dataloader, device, desc, max_length, warmup):
    """Greedy generate + measure latency."""
    m.eval()
    preds_dict = {}
    latencies  = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc=desc)):
            px = batch['pixel_values'].to(device)

            t0  = time.perf_counter()
            ids = m.generate(
                pixel_values=px,
                max_length=max_length,
                num_beams=1,
                do_sample=False,
            )
            t1  = time.perf_counter()

            if batch_idx >= warmup:
                latencies.append(t1 - t0)

            texts = processor.batch_decode(ids, skip_special_tokens=True)
            for img_f, text in zip(batch['image_file'], texts):
                preds_dict[img_f] = text.strip()

    return preds_dict, latencies


def make_benchmark_record(name, backend, precision, device_name,
                           preds_dict, refs_dict, latencies,
                           extra=None):
    metrics    = calculate_metrics(preds_dict, refs_dict)
    avg_lat    = float(np.mean(latencies))    if latencies else 0.0
    p95_lat    = float(np.percentile(latencies, 95)) if latencies else 0.0
    throughput = float(BATCH_SIZE / avg_lat)  if avg_lat > 0 else 0.0
    rec = {
        'name':                           name,
        'backend':                        backend,
        'precision':                      precision,
        'device':                         device_name,
        'provider':                       'torch',
        'generation_strategy':            'greedy',
        'cache_enabled':                  True,
        'batch_size':                     BATCH_SIZE,
        'max_test_images':                MAX_TEST_IMAGES,
        'avg_latency_seconds_per_batch':  avg_lat,
        'p95_latency_seconds_per_batch':  p95_lat,
        'throughput_images_per_second':   throughput,
        'bleu4':                          metrics['bleu4'],
        'rougeL':                         metrics['rougeL'],
        'meteor':                         metrics['meteor'],
        'num_images':                     metrics['num_images'],
    }
    if extra:
        rec.update(extra)
    return rec


print("Utilities defined.")


Utilities defined.


In [4]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
fp32_model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
fp32_model.eval()

test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

refs_dict = build_references(test_loader)
print(f"Test images: {len(test_dataset)}  |  batches: {len(test_loader)}")


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 3450.81it/s]


Test images: 200  |  batches: 13


In [5]:
fp32_preds, fp32_latencies = run_generation(
    fp32_model, processor, test_loader, device,
    desc='[1/2] FP32 baseline (greedy)', max_length=MAX_LENGTH, warmup=WARMUP_STEPS
)

fp32_benchmark = make_benchmark_record(
    'baseline_fp32_greedy', 'pytorch', 'fp32', device.type,
    fp32_preds, refs_dict, fp32_latencies,
    extra={'note': 'greedy decode, used as fair baseline for INT8 comparison'}
)
print(json.dumps(fp32_benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'baseline_fp32_greedy.json'), 'w', encoding='utf-8') as f:
    json.dump(fp32_benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/baseline_fp32_greedy.json")


[1/2] FP32 baseline (greedy): 100%|██████████| 13/13 [03:53<00:00, 18.00s/it]
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{
  "name": "baseline_fp32_greedy",
  "backend": "pytorch",
  "precision": "fp32",
  "device": "cpu",
  "provider": "torch",
  "generation_strategy": "greedy",
  "cache_enabled": true,
  "batch_size": 16,
  "max_test_images": 200,
  "avg_latency_seconds_per_batch": 17.800532730000487,
  "p95_latency_seconds_per_batch": 20.34350868499805,
  "throughput_images_per_second": 0.8988495031406604,
  "bleu4": 22.826964076992915,
  "rougeL": 49.84811186344499,
  "meteor": 35.336904377949104,
  "num_images": 200,
  "note": "greedy decode, used as fair baseline for INT8 comparison"
}

Saved: results/baseline_fp32_greedy.json


In [6]:
int8_model = copy.deepcopy(fp32_model)
int8_model.eval()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    int8_model.text_decoder = torch.quantization.quantize_dynamic(
        int8_model.text_decoder,
        {torch.nn.Linear},
        dtype=torch.qint8
    )
int8_model.eval()

n_qdynamic = sum(
    1 for _, mod in int8_model.text_decoder.named_modules()
    if 'quantized' in type(mod).__module__
)
print(f'Quantized linear layers : {n_qdynamic}')

def param_mb(mod):
    return sum(p.numel() * p.element_size() for p in mod.parameters()) / 1e6

fp32_dec_mb = param_mb(fp32_model.text_decoder)
int8_dec_mb = param_mb(int8_model.text_decoder)
fp32_vis_mb = param_mb(fp32_model.vision_model)

print(f'Vision encoder FP32     : {fp32_vis_mb:.1f} MB  (unchanged)')
print(f'Text decoder FP32       : {fp32_dec_mb:.1f} MB')
print(f'Text decoder INT8       : {int8_dec_mb:.1f} MB')
print('(torch.quantization INT8: weight data la INT8, activation FP32)')

torch.save(int8_model.state_dict(), os.path.join(SAVE_DIR, 'pytorch_int8_state_dict.pt'))
processor.save_pretrained(SAVE_DIR)
print('Saved INT8 artifacts:', SAVE_DIR)


Quantized linear layers : 244
Vision encoder FP32     : 344.4 MB  (unchanged)
Text decoder FP32       : 654.5 MB
Text decoder INT8       : 198.7 MB
(torch.quantization INT8: weight data la INT8, activation FP32)
Saved INT8 artifacts: ./model_quantized/pytorch_int8


In [7]:
int8_preds, int8_latencies = run_generation(
    int8_model, processor, test_loader, device,
    desc='[2/2] INT8 quantized (decoder-only)', max_length=MAX_LENGTH, warmup=WARMUP_STEPS
)

int8_benchmark = make_benchmark_record(
    'pytorch_int8', 'pytorch', 'int8', device.type,
    int8_preds, refs_dict, int8_latencies,
    extra={
        'quantization_lib':   'torchao',
        'quantization_method':'int8_weight_only',
        'quantization_scope': 'decoder_only',
    }
)
print(json.dumps(int8_benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'pytorch_int8.json'), 'w', encoding='utf-8') as f:
    json.dump(int8_benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/pytorch_int8.json")


[2/2] INT8 quantized (decoder-only): 100%|██████████| 13/13 [05:42<00:00, 26.32s/it]
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{
  "name": "pytorch_int8",
  "backend": "pytorch",
  "precision": "int8",
  "device": "cpu",
  "provider": "torch",
  "generation_strategy": "greedy",
  "cache_enabled": true,
  "batch_size": 16,
  "max_test_images": 200,
  "avg_latency_seconds_per_batch": 26.05338781999999,
  "p95_latency_seconds_per_batch": 35.5301332549996,
  "throughput_images_per_second": 0.6141235877092934,
  "bleu4": 15.552823881473095,
  "rougeL": 46.20819496977278,
  "meteor": 34.18083590044007,
  "num_images": 200,
  "quantization_lib": "torchao",
  "quantization_method": "int8_weight_only",
  "quantization_scope": "decoder_only"
}

Saved: results/pytorch_int8.json


In [8]:
rows = [fp32_benchmark, int8_benchmark]
cols = ['name', 'precision', 'bleu4', 'rougeL', 'meteor',
        'avg_latency_seconds_per_batch', 'throughput_images_per_second']
df   = pd.DataFrame(rows)[cols]

fp32_lat = df.loc[df['precision']=='fp32', 'avg_latency_seconds_per_batch'].values[0]
df['speedup_vs_fp32'] = (fp32_lat / df['avg_latency_seconds_per_batch']).round(2)

fp32_bleu = df.loc[df['precision']=='fp32', 'bleu4'].values[0]
df['bleu4_drop'] = (df['bleu4'] - fp32_bleu).round(2)

print(df.to_string(index=False))


                name precision     bleu4    rougeL    meteor  avg_latency_seconds_per_batch  throughput_images_per_second  speedup_vs_fp32  bleu4_drop
baseline_fp32_greedy      fp32 22.826964 49.848112 35.336904                      17.800533                      0.898850             1.00        0.00
        pytorch_int8      int8 15.552824 46.208195 34.180836                      26.053388                      0.614124             0.68       -7.27
